In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.onnx
X = torch.randn(5, 40)
Y = torch.randint(0, 2, (5, 1)).float()  # True labels (0 or 1)

model = nn.Sequential(
    nn.Linear(40, 16),  #  (16 x 40)
    nn.ReLU(),
    nn.Linear(16, 10),  # (10 x 16)
    nn.ReLU(),
    nn.Linear(10, 1)    # (1 x 10)
   
)


criterion = nn.BCEWithLogitsLoss()


optimizer = optim.SGD(model.parameters(), lr=0.1)


logits = model(X)           
loss = criterion(logits, Y) 


optimizer.zero_grad()       
loss.backward()             


optimizer.step()            


print(f"Loss after Iteration 1: {loss.item():.4f}")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import sys

torch.manual_seed(42)
X = torch.randn(100, 40)
Y = torch.randint(0, 2, (100, 1)).float()  # True labels: 0 or 1


model = nn.Sequential(
    nn.Linear(40, 16),  # Layer 1: 40 features -> 16 neurons
    nn.ReLU(),          # Activation 1
    nn.Linear(16, 10),  # Layer 2: 16 neurons -> 10 neurons
    nn.ReLU(),          # Activation 2
    nn.Linear(10, 1)    # Layer 3 (Output): 10 neurons -> 1 output
)


criterion = nn.BCEWithLogitsLoss()


optimizer = optim.SGD(model.parameters(), lr=0.1)

epochs = 10000

for epoch in range(epochs):
   
    logits = model(X)           
    
    
    loss = criterion(logits, Y) 
    
    
    optimizer.zero_grad()       
    
    loss.backward()             
    
    
    optimizer.step()            
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:3d} | Loss: {loss.item():.4f}")

# Optional: Inspecting the calculated gradients for Layer 1's weights
print("\nShape of Layer 1 Weight Matrix (W1):", model[0].weight.shape)
print("Shape of Layer 1 Gradients (dW1):", model[0].weight.grad.shape)

In [ ]:
import pandas as pd


In [ ]:
df = pd.read_csv("mod_data.csv")

In [ ]:
df.head(10)

In [ ]:
import re
from tqdm import tqdm
from functools import lru_cache
from spellchecker import SpellChecker

spell = SpellChecker()
custom_words = ['adhd', 'ptsd', 'crps', 'agender', 'lgbt', 'lgbts', 'idk', 'lol']
spell.word_frequency.load_words(custom_words)
tqdm.pandas()
# @lru_cache(maxsize=50000)
# def fix_word(word):
#     if "'" in word or len(word) <= 2:
#         return word
#     return spell.correction(word) or word

def clean_and_spellcheck(text):
    if not isinstance(text, str):
        return []
    
    # Separate glued letters and numbers (e.g., '2929\n\n\nThat's' or 'mental14')
    text = re.sub(r'([a-zA-Z]+)(\d+)', r'\1 \2', text)
    
    # Normalize curly apostrophes to straight apostrophes
    text = text.replace("’", "'")
    
    # Extract clean alphabetic tokens
    clean_tokens = re.findall(r"[a-zA-Z]+(?:'[a-zA-Z]+)?", text.lower())
    
    return clean_tokens

df['clean_tokens'] = df['text'].progress_apply(clean_and_spellcheck)

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
import joblib
import torch


df['clean_token_text'] = df['clean_tokens'].str.join(' ')
texts = df['clean_token_text']
targets = df['target'].values


vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), min_df=2)

X_tfidf = vectorizer.fit_transform(texts)

X_tensor = torch.tensor(X_tfidf.toarray(), dtype=torch.float32)
Y_tensor = torch.tensor(targets, dtype=torch.float32).view(-1, 1)

X_train, X_test, Y_train, Y_test = train_test_split(X_tensor, Y_tensor, test_size=0.2, random_state=42)


train_dataset = TensorDataset(X_train, Y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

input_size = X_train.shape[1] 

model = nn.Sequential(
    nn.Linear(input_size, 16), 
    nn.ReLU(),
    nn.Linear(16, 10),         
    nn.ReLU(),
    nn.Linear(10, 1)           
)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001) 


epochs = 55

for epoch in tqdm(range(epochs)):
    model.train()
    running_loss = 0.0
    
    for batch_X, batch_Y in train_loader:
        logits = model(batch_X)
        loss = criterion(logits, batch_Y)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
    
    avg_loss = running_loss / len(train_loader)
    
    print(f"Epoch {epoch+1:2d}/{epochs} | Average Loss: {avg_loss:.4f}")
# exported_program = torch.export.export(model, example_inputs)
torch.save(model.state_dict(), "suicide_classifier.pth")
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")


In [ ]:

model.eval()


with torch.no_grad():

    test_logits = model(X_test)
    
    test_probabilities = torch.sigmoid(test_logits)

    test_predictions = (test_probabilities >= 0.5).float()
    
    
    correct_matches = (test_predictions == Y_test).sum().item()
    total_samples = Y_test.size(0)
    
    accuracy = (correct_matches / total_samples) * 100
    
    print(f"Validation Accuracy: {accuracy:.2f}%")

In [ ]:
import joblib
import torch

# Load vectorizer
vectorizer = joblib.load("tfidf_vectorizer.pkl")

# Load model
model.load_state_dict(torch.load("suicide_classifier.pth", weights_only=True))
model.eval()

new_text = ["""
 am new to steam deck, and i really hate flatpack or appimage and i want to use pacman and yay, is it safe and is there any thing that i should keep in mind, i don't know where i read that but someone was saying that if i did that and i updated the steam deck it will break or smt.

and thx 
"""]

# Transform text into TF-IDF
X_new = vectorizer.transform(new_text)

# Convert sparse matrix into PyTorch tensor
X_new_tensor = torch.tensor(X_new.toarray(), dtype=torch.float32)

# Prediction
with torch.no_grad():
    logits = model(X_new_tensor)
    probabilities = torch.sigmoid(logits)
    predictions = (probabilities >= 0.5).float()

print("Probability:", probabilities.item())
print("Prediction:", predictions.item())

In [ ]:
df['clean_token_text'] = df['clean_tokens'].str.join(' ')
texts = df['clean_token_text']
targets = df['target'].values


vectorizer = TfidfVectorizer(max_features=15000, ngram_range=(1, 2))

X_tfidf = vectorizer.fit_transform(texts)
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")
